# Best Diversifying Pair in the S&P 500

Searches every pair of S&P 500 stocks (about 122,000 pairs) for the equal-weight portfolio with the strongest and most stable **diversification gain**.

**Method**
1. Download prices and compute daily log returns for 2024-01-02 to 2025-12-31 (stocks with at least 90% data coverage).
2. For each pair, build a 50/50 portfolio and compute a rolling 63-day diversification gain:
   `DG = 1 - portfolio volatility / (0.5 * vol_A + 0.5 * vol_B)`
3. Score each pair as `mean(DG) - 0.5 * std(DG)`, which rewards a high and stable gain.
4. Report the top pair overall, and the top pair whose annualised return is at least that of the S&P 500 index (`^GSPC`).

The brute-force search over all pairs takes roughly 7 minutes.


In [ ]:
# ============================================================
# STEP 1: Get the list of S&P 500 stock tickers
# ============================================================
import pandas as pd            # pandas: tables (DataFrames) and data analysis
import requests                # requests: download web pages
from io import StringIO        # StringIO: lets pandas read a string as if it were a file

# Wikipedia page that contains a table of every S&P 500 company
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Download the page's HTML as text.
# The User-Agent header pretends to be a normal browser, because Wikipedia
# may reject requests that look like plain scripts.
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text

# pd.read_html finds every table in the HTML and returns them as a list of DataFrames.
#   [0]         -> take the first table (the companies table)
#   ["Symbol"]  -> keep only the ticker column
#   .tolist()   -> convert that column into a normal Python list
sp500_tickers = pd.read_html(StringIO(html))[0]["Symbol"].tolist()

# Yahoo Finance writes class-B shares with a dash instead of a dot
# (Wikipedia: BRK.B  ->  Yahoo: BRK-B). Replace every "." with "-".
sp500_tickers = [ticker.replace(".", "-") for ticker in sp500_tickers]

# Show the list (last line of a notebook cell is displayed automatically)
sp500_tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BE',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'COP',
 'E

In [ ]:
# ============================================================
# STEP 2: Download historical prices
# ============================================================
import numpy as np             # numpy: fast math (we use np.log and np.inf)
import pandas as pd
import yfinance as yf          # yfinance: downloads prices from Yahoo Finance

# Download daily prices for all ~500 stocks, plus SPY.
# SPY (the S&P 500 ETF) is added only as a reference to count trading days later.
prices = yf.download(
    sp500_tickers + ["SPY"],   # list of everything to download
    start="2023-12-26",        # start a few days early so the first return (2024-01-02) can be computed,
                               #   because a return needs the PREVIOUS day's price
    end="2026-01-01",          # end date is exclusive, so this includes 2025-12-31
    auto_adjust=True,          # adjust prices for splits and dividends (fairer returns)
    group_by="column",         # columns are grouped by field (Close, Open...) then ticker
    threads=True,              # download several tickers in parallel (faster)
    progress=True,             # show a progress bar
)

# Keep only the closing prices.
# Result: rows = dates, columns = tickers.
close = prices["Close"]

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['HONA', 'VYLR', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2023-12-26 -> 2026-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1703566800, endDate = 1767243600")')


In [ ]:
# ============================================================
# STEP 3: Log returns + 90% coverage filter
# ============================================================
START, END = "2024-01-02", "2025-12-31"
window = 63

rets_all = np.log(close / close.shift(1)).replace([np.inf, -np.inf], np.nan).loc[START:END]
sessions = rets_all["SPY"].dropna().index          # US trading sessions
returns_log = rets_all.loc[sessions].drop(columns="SPY")
returns_log = returns_log.loc[:, returns_log.notna().mean() >= 0.90]
print(returns_log.shape)

# ============================================================
# STEP 4: S&P 500 benchmark (^GSPC), same dates and method
# ============================================================
gspc = yf.download("^GSPC", start="2023-12-26", end="2026-01-01",
                   auto_adjust=True, progress=False)["Close"].squeeze()
gspc_ret = np.log(gspc / gspc.shift(1)).reindex(sessions)
gspc_ann = 252 * gspc_ret.mean()
print(f"S&P 500 annualized mean log return: {gspc_ann:.2%}")

# ============================================================
# STEP 5: Diversification-gain statistics for one pair
# ============================================================
def pair_stats(a, b):
    pair = returns_log[[a, b]]
    pair = pair.where(pair.notna().all(axis=1))     # keep only days where BOTH have a return

    port = pair @ np.array([0.5, 0.5])              # NaN if either is missing
    vol_p = port.rolling(window).std()              # NaN unless all 63 days are present
    vol_s = pair.rolling(window).std()
    dg = (1 - vol_p / (0.5 * vol_s[a] + 0.5 * vol_s[b])).dropna()

    return {
        "A": a, "B": b,
        "ann_ret": 252 * port.mean(),               # annualized mean portfolio log return
        "mean_DG": dg.mean(),
        "sd_DG": dg.std(),                          # ddof=1 by default in pandas
        "min_DG": dg.min(),
        "score": dg.mean() - 0.5 * dg.std(),
        "n_windows": len(dg),
    }

# ============================================================
# STEP 6: Brute force over all pairs
# ============================================================
from itertools import combinations
from tqdm import tqdm

results = [pair_stats(a, b)
           for a, b in tqdm(list(combinations(returns_log.columns, 2)))]
res = pd.DataFrame(results).dropna(subset=["score"])

# Convert back to Wikipedia symbols (BRK-B -> BRK.B) and sort each pair alphabetically
res[["A", "B"]] = res[["A", "B"]].apply(lambda s: s.str.replace("-", "."))
res[["A", "B"]] = np.sort(res[["A", "B"]].to_numpy(), axis=1)

# ============================================================
# STEP 7: Best pair overall (A) and best pair with return >= S&P 500 (B)
# ============================================================
partA = res.sort_values("score", ascending=False).iloc[0]
partB = res[res["ann_ret"] >= gspc_ann].sort_values("score", ascending=False).iloc[0]

summary = pd.DataFrame({"Part A": partA, "Part B": partB}).T
for c in ["ann_ret", "mean_DG", "sd_DG", "min_DG", "score"]:
    summary[c] = (summary[c].astype(float) * 100).round(2)     # in %
summary

(502, 495)
S&P 500 annualized mean log return: 18.14%


100%|██████████| 122265/122265 [07:12<00:00, 282.89it/s]


,A,B,ann_ret,mean_DG,sd_DG,min_DG,score,n_windows
Part A,AMZN,CBOE,19.49,40.11,5.75,24.8,37.23,440
Part B,AMZN,CBOE,19.49,40.11,5.75,24.8,37.23,440


In [ ]:
# ============================================================
# STEP 8: Results
# ============================================================
print("Pairs with annualised return >= S&P 500:", (res["ann_ret"] >= gspc_ann).sum())
print("Best constrained pair annualised return vs S&P 500:", f"{partB.ann_ret:.2%} vs {gspc_ann:.2%}")

print("\nTop 5 pairs overall:")
display(res.sort_values("score", ascending=False).head(5))
print("Top 5 pairs with return >= S&P 500:")
display(res[res["ann_ret"] >= gspc_ann].sort_values("score", ascending=False).head(5))


Pairs with annualised return >= S&P 500: 37269
Best constrained pair annualised return vs S&P 500: 19.49% vs 18.14%

Top 5 pairs overall:


,A,B,ann_ret,mean_DG,sd_DG,min_DG,score,n_windows
14895,AMZN,CBOE,0.194877,0.401089,0.057533,0.248019,0.372322,440
35657,CBOE,GOOG,0.292718,0.371848,0.040972,0.272195,0.351362,440
35680,CBOE,HUBB,0.171708,0.372774,0.045553,0.291766,0.349998,440
35766,CBOE,MSFT,0.156788,0.372144,0.045502,0.285561,0.349393,440
35658,CBOE,GOOGL,0.294309,0.368444,0.041157,0.266302,0.347866,440


Top 5 pairs with return >= S&P 500:


,A,B,ann_ret,mean_DG,sd_DG,min_DG,score,n_windows
14895,AMZN,CBOE,0.194877,0.401089,0.057533,0.248019,0.372322,440
35657,CBOE,GOOG,0.292718,0.371848,0.040972,0.272195,0.351362,440
35658,CBOE,GOOGL,0.294309,0.368444,0.041157,0.266302,0.347866,440
34695,CAT,CBOE,0.263690,0.357831,0.044296,0.250596,0.335683,440
35884,CBOE,TRMB,0.187089,0.363590,0.059802,0.241005,0.333689,440
